# Accounts Receivable Portfolio Analysis

Exploratory analysis on the **Gold layer** produced by the pipeline (`python -m ar_pipeline.run`).

> All data is **synthetic**. Company names are fictional; no real customer or company information is used.

**Questions answered**
1. How large is the portfolio and how much of it is overdue?
2. Is portfolio quality improving or deteriorating?
3. Which segments, companies and collectors concentrate the risk?
4. Once a customer falls behind, how likely is it to recover?
5. Where should the collections team focus next month?

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.options.display.float_format = '{:,.2f}'.format
GOLD = Path('..') / 'data' / 'gold'
con = duckdb.connect()
for table in GOLD.glob('*.parquet'):
    con.execute(f"CREATE VIEW {table.stem} AS SELECT * FROM read_parquet('{table.as_posix()}')")
con.sql('SHOW TABLES').df()

,name
0,agg_portfolio_monthly
1,bucket_transitions
2,customer_monthly
3,dim_aging_bucket
4,dim_business_area
5,dim_company
6,dim_customer
7,dim_date
8,fact_ar_open_items


## 1. Portfolio size and overdue share

In [2]:
monthly = con.sql('''
    SELECT snapshot_date,
           round(sum(amount_lc) / 1e6, 1)                                     AS total_m,
           round(sum(amount_lc) FILTER (WHERE is_overdue) / 1e6, 1)           AS overdue_m,
           round(sum(amount_lc) FILTER (WHERE is_overdue) / sum(amount_lc), 3) AS overdue_ratio,
           round(sum(amount_lc) FILTER (WHERE aging_bucket_id = 5) / sum(amount_lc), 3) AS over_90_ratio,
           count(DISTINCT customer_id)                                        AS customers
    FROM fact_ar_open_items GROUP BY 1 ORDER BY 1
''').df()
monthly

,snapshot_date,total_m,overdue_m,overdue_ratio,over_90_ratio,customers
0,2025-01-31,103.40,26.30,0.25,0.02,733
1,2025-02-28,108.90,24.40,0.22,0.02,760
2,2025-03-31,106.00,34.50,0.33,0.03,736
3,2025-04-30,104.30,29.40,0.28,0.04,723
4,2025-05-31,106.30,30.80,0.29,0.05,736
5,2025-06-30,112.70,33.20,0.29,0.04,736
6,2025-07-31,113.40,34.90,0.31,0.05,713
7,2025-08-31,115.50,36.90,0.32,0.06,702
8,2025-09-30,119.00,42.80,0.36,0.06,704
9,2025-10-31,117.20,43.80,0.37,0.07,699


**Reading:** the open balance grows moderately during the year, but the **overdue share rises much faster** and the >90-day bucket roughly quadruples its weight. Growth in receivables is coming from late payment, not from more sales.

## 2. Risk by customer segment (latest month)

In [3]:
con.sql('''
    SELECT d.segment,
           count(DISTINCT f.customer_id)                                        AS customers,
           round(sum(f.amount_lc) / 1e6, 1)                                     AS balance_m,
           round(sum(f.amount_lc) FILTER (WHERE f.is_overdue) / sum(f.amount_lc), 3) AS overdue_ratio,
           round(sum(f.amount_lc) FILTER (WHERE f.aging_bucket_id = 5) / sum(f.amount_lc), 3) AS over_90_ratio
    FROM fact_ar_open_items f JOIN dim_customer d USING (customer_id)
    WHERE f.snapshot_date = (SELECT max(snapshot_date) FROM fact_ar_open_items)
    GROUP BY 1 ORDER BY over_90_ratio DESC
''').df()

,segment,customers,balance_m,overdue_ratio,over_90_ratio
0,SME,301,34.60,0.64,0.14
1,Retail,163,5.40,0.69,0.14
2,Government,82,20.10,0.46,0.10
3,Corporate,142,58.00,0.26,0.03


## 3. Concentration: how much of the overdue balance sits with a few customers?

In [4]:
pareto = con.sql('''
    WITH latest AS (
        SELECT c.customer_id, d.customer_name, d.segment, c.overdue_balance_lc, c.worst_bucket_id, c.trend_status
        FROM customer_monthly c JOIN dim_customer d USING (customer_id)
        WHERE c.snapshot_date = (SELECT max(snapshot_date) FROM customer_monthly) AND c.overdue_balance_lc > 0
    )
    SELECT *,
           sum(overdue_balance_lc) OVER (ORDER BY overdue_balance_lc DESC) / sum(overdue_balance_lc) OVER () AS cumulative_share,
           row_number() OVER (ORDER BY overdue_balance_lc DESC) AS rank
    FROM latest ORDER BY overdue_balance_lc DESC
''').df()
customers_for_80 = int((pareto['cumulative_share'] <= 0.80).sum()) + 1
print(f'{customers_for_80} of {len(pareto)} overdue customers ({customers_for_80/len(pareto):.0%}) hold 80% of the overdue balance')
pareto.head(10)

109 of 418 overdue customers (26%) hold 80% of the overdue balance


,customer_id,customer_name,segment,overdue_balance_lc,worst_bucket_id,trend_status,cumulative_share,rank
0,C000204,NORTE TURISMO E.I.R.L. 0204,Corporate,"1,521,409.16",3,Deteriorated,0.03,1
1,C000153,PACIFICO CARGO S.R.L. 0153,Corporate,"1,181,431.30",5,Stable,0.05,2
2,C000554,SIERRA MINERIA E.I.R.L. 0554,Government,"1,068,289.99",5,Deteriorated,0.08,3
3,C000623,SIERRA AGROINDUSTRIAS E.I.R.L. 0623,Government,"1,050,483.93",5,Stable,0.10,4
4,C000636,MISTI INVERSIONES S.A.C. 0636,Corporate,"969,917.88",5,Stable,0.12,5
5,C000711,PACIFICO INGENIERIA E.I.R.L. 0711,Government,"938,109.89",4,Improved,0.13,6
6,C000269,COSTA CARGO S.A. 0269,Corporate,"845,754.16",3,Improved,0.15,7
7,C000681,SELVA DISTRIBUIDORA S.A. 0681,Corporate,"775,804.93",3,Deteriorated,0.17,8
8,C000511,LLAMA MINERIA E.I.R.L. 0511,Corporate,"743,714.29",3,Deteriorated,0.18,9
9,C000762,CONDOR CARGO S.A. 0762,SME,"725,262.11",5,Stable,0.20,10


## 4. Collector workload and portfolio quality

In [5]:
con.sql('''
    SELECT d.collector,
           count(DISTINCT f.customer_id)                                         AS customers,
           round(sum(f.amount_lc) FILTER (WHERE f.is_overdue) / 1e6, 2)          AS overdue_m,
           round(sum(f.amount_lc) FILTER (WHERE f.is_overdue) / sum(f.amount_lc), 3) AS overdue_ratio
    FROM fact_ar_open_items f JOIN dim_customer d USING (customer_id)
    WHERE f.snapshot_date = (SELECT max(snapshot_date) FROM fact_ar_open_items)
    GROUP BY 1 ORDER BY overdue_m DESC
''').df()

,collector,customers,overdue_m,overdue_ratio
0,COL-03,111,10.01,0.58
1,COL-06,105,9.90,0.42
2,COL-01,116,9.02,0.40
3,COL-05,123,8.46,0.39
4,COL-02,111,7.65,0.44
5,COL-04,122,4.94,0.30


## 5. Roll rates: what happens one month after a customer falls behind?

In [6]:
transitions = con.sql('''
    SELECT from_bucket_id, to_bucket_id, sum(customers) AS customers
    FROM bucket_transitions GROUP BY ALL
''').df()
labels = dict(con.sql('SELECT aging_bucket_id, aging_bucket FROM dim_aging_bucket').fetchall())
labels[0] = 'Settled'
matrix = transitions.pivot_table(index='from_bucket_id', columns='to_bucket_id', values='customers', fill_value=0)
matrix = matrix.div(matrix.sum(axis=1), axis=0).rename(index=labels, columns=labels)
matrix.style.format('{:.0%}').background_gradient(cmap='Blues', axis=None)

to_bucket_id,Settled,Not due,0-30,31-60,61-90,>90
from_bucket_id,,,,,,
Not due,15%,48%,38%,0%,0%,0%
0-30,11%,30%,39%,19%,0%,0%
31-60,5%,6%,25%,46%,18%,0%
61-90,1%,4%,12%,36%,33%,13%
>90,0%,0%,0%,1%,2%,97%


**Reading:** customers in 0-30 days still recover often, but the probability of recovery falls sharply with each bucket. Once a customer reaches **>90 days it almost never comes back** within a month, so the cheapest moment to act is the 31-60 bucket.

## 6. Focus list for next month

In [7]:
con.sql('''
    SELECT d.customer_name, d.segment, d.collector, c.worst_bucket_id,
           round(c.overdue_balance_lc, 0) AS overdue_balance_lc, c.trend_status
    FROM customer_monthly c JOIN dim_customer d USING (customer_id)
    WHERE c.snapshot_date = (SELECT max(snapshot_date) FROM customer_monthly)
      AND c.worst_bucket_id IN (3, 4)          -- 31-90 days: still recoverable
      AND c.trend_status = 'Deteriorated'
    ORDER BY c.overdue_balance_lc DESC
    LIMIT 15
''').df()

,customer_name,segment,collector,worst_bucket_id,overdue_balance_lc,trend_status
0,NORTE TURISMO E.I.R.L. 0204,Corporate,COL-06,3,"1,521,409.00",Deteriorated
1,SELVA DISTRIBUIDORA S.A. 0681,Corporate,COL-05,3,"775,805.00",Deteriorated
2,LLAMA MINERIA E.I.R.L. 0511,Corporate,COL-05,3,"743,714.00",Deteriorated
3,LLAMA INVERSIONES S.R.L. 0086,Corporate,COL-05,3,"591,951.00",Deteriorated
4,MISTI INVERSIONES E.I.R.L. 0171,SME,COL-05,3,"558,990.00",Deteriorated
5,HUASCAR TURISMO S.R.L. 0485,Corporate,COL-06,3,"543,946.00",Deteriorated
6,PACIFICO INGENIERIA S.A. 0562,SME,COL-05,3,"490,342.00",Deteriorated
7,CUMBRE INVERSIONES S.A. 0209,Corporate,COL-02,3,"471,780.00",Deteriorated
8,SUR INGENIERIA S.A. 0474,Corporate,COL-01,3,"454,275.00",Deteriorated
9,NORTE INGENIERIA S.A.C. 0592,SME,COL-05,4,"450,140.00",Deteriorated


## Conclusions

* Portfolio growth is driven by **slower payment**, not by sales: the overdue share and the >90-day share rise throughout the year in all three companies.
* Risk is **concentrated**: a minority of customers holds most of the overdue balance, so a short focus list covers most of the money at risk.
* **Timing matters more than volume**: recovery probability collapses after 90 days. Prioritising deteriorating customers in the 31-90 day range gives the best return on collection effort.

*Synthetic data: the patterns are realistic by design, but the figures do not describe any real company.*